# Train All Models: 4 Attention Mechanisms x 2 Datasets (50% Dataset)

Unified training notebook for retraining all models with RoPE from scratch.

**Models:**
- **MHA-RoPE**: Multi-Head Attention with Rotary Position Embeddings
- **MQA-RoPE**: Multi-Query Attention with RoPE
- **GQA-4-RoPE**: Grouped-Query Attention (4 KV heads) with RoPE
- **MLA**: Multi-Head Latent Attention with Decoupled RoPE

**Datasets:** TinyStories, SimpleStories

**Config:** 256d, 4 layers, 8 heads, vocab 50257, ~16M params each

**Training:** 6000 steps, batch 64 (effective 256), lr 5e-5, BFloat16, cosine schedule

**H100 Optimized:** TF32 enabled, ~35 min per model, ~4.5 hours total

**Dataset:** 50% of each dataset (computed at runtime)

**Output:** 8 checkpoints saved to Google Drive

## 1. Setup & H100 Optimization

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.2f} GB")
    print(f"BFloat16 supported: {torch.cuda.is_bf16_supported()}")

# H100-specific optimizations
torch.set_float32_matmul_precision('high')  # Enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("\nTF32 optimizations enabled for H100")

In [ ]:
# Install dependencies
!pip install -q transformers datasets tqdm tensorboard

In [ ]:
# Clone repository
import os
if os.path.exists('PROJECT'):
    !rm -rf PROJECT
    print("Existing repo removed")
!git clone -b feat/mla https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
print("Repository cloned")
%cd PROJECT

In [ ]:
# Clear cache, patch __init__.py, import all 4 transformer modules
import sys, os, importlib, shutil, torch, gc

print("Clearing Python cache...")
modules_to_remove = [m for m in list(sys.modules.keys())
                     if any(x in m for x in ['mla', 'mqa', 'gqa', 'mha', 'train', 'transformer', 'data_loader', 'attention', 'layers'])]
for module in modules_to_remove:
    del sys.modules[module]
print(f"Removed {len(modules_to_remove)} cached modules")

cache_dirs = [
    '/content/PROJECT/AttentionHeads/mha/__pycache__',
    '/content/PROJECT/AttentionHeads/mqa/__pycache__',
    '/content/PROJECT/AttentionHeads/gqa/__pycache__',
    '/content/PROJECT/AttentionHeads/mla/__pycache__',
    '/content/PROJECT/AttentionHeads/__pycache__',
]
for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

project_root = '/content/PROJECT'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch __init__.py to comment out positional_encoding import
init_file_path = os.path.join(project_root, 'AttentionHeads', 'mha', '__init__.py')
if os.path.exists(init_file_path):
    with open(init_file_path, 'r') as f:
        lines = f.readlines()
    patched_lines = []
    modified = False
    in_block = False
    block_indent = -1
    for line in lines:
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())
        if "from .positional_encoding import" in line:
            in_block = True
            block_indent = indent
            if not stripped.startswith('#'):
                patched_lines.append(f"# PATCHED: {stripped}\n")
                modified = True
            else:
                patched_lines.append(line)
            continue
        if in_block:
            if indent > block_indent or stripped == '' or (')' in stripped and indent <= block_indent):
                if not stripped.startswith('#'):
                    patched_lines.append(f"# PATCHED: {stripped}\n")
                    modified = True
                else:
                    patched_lines.append(line)
                if ')' in stripped and indent <= block_indent:
                    in_block = False
                continue
            else:
                in_block = False
        patched_lines.append(line)
    if modified:
        with open(init_file_path, 'w') as f:
            f.writelines(patched_lines)
        print("Patched __init__.py")

# Import all 4 transformer modules
from AttentionHeads.mha import transformer as mha_transformer
from AttentionHeads.mqa import transformer as mqa_transformer
from AttentionHeads.gqa import transformer as gqa_transformer
from AttentionHeads.mla import transformer as mla_transformer
from AttentionHeads.mha import data_loader, train

for mod in [mha_transformer, mqa_transformer, gqa_transformer, mla_transformer, data_loader, train]:
    importlib.reload(mod)

GPTNeoTrainer = train.GPTNeoTrainer
print("All modules imported successfully!")

## 2. Configuration

In [ ]:
# Pre-compute 50% dataset sizes
from datasets import load_dataset_builder

DATASET_SIZES = {}

print("Computing 50% dataset sizes...")
ts_builder = load_dataset_builder('roneneldan/TinyStories')
ts_train_total = ts_builder.info.splits['train'].num_examples
ts_val_total = ts_builder.info.splits['validation'].num_examples
DATASET_SIZES['tinystories'] = {'train': ts_train_total // 2, 'val': ts_val_total // 2}
print(f"  TinyStories: {DATASET_SIZES['tinystories']['train']:,} train, {DATASET_SIZES['tinystories']['val']:,} val (of {ts_train_total:,} / {ts_val_total:,} total)")

ss_builder = load_dataset_builder('SimpleStories/SimpleStories')
ss_train_total = ss_builder.info.splits['train'].num_examples
ss_val_total = ss_builder.info.splits['test'].num_examples
DATASET_SIZES['simplestories'] = {'train': ss_train_total // 2, 'val': ss_val_total // 2}
print(f"  SimpleStories: {DATASET_SIZES['simplestories']['train']:,} train, {DATASET_SIZES['simplestories']['val']:,} val (of {ss_train_total:,} / {ss_val_total:,} total)")

print("50% dataset sizes computed!")


def get_training_config(model_name, architecture, dataset, log_dir, checkpoint_dir):
    """Create configuration dictionary for training.

    Args:
        model_name: Name for logging (e.g. 'gptneo_mha_ts')
        architecture: Architecture type (e.g. 'gpt_neo_mha')
        dataset: 'tinystories' or 'simplestories'
        log_dir: TensorBoard log directory
        checkpoint_dir: Model checkpoint directory
    """
    # Dataset-specific settings
    if dataset == 'tinystories':
        dataset_name = 'roneneldan/TinyStories'
        text_column = 'text'
        val_split = 'validation'
        ds_short = 'ts'
    else:
        dataset_name = 'SimpleStories/SimpleStories'
        text_column = 'story'
        val_split = 'test'
        ds_short = 'ss'

    # Use pre-computed 50% dataset sizes
    train_samples = DATASET_SIZES[dataset]['train']
    val_samples = DATASET_SIZES[dataset]['val']

    return {
        "model_name": model_name,
        "architecture": architecture,
        "model": {
            "vocab_size": 50257,
            "hidden_size": 256,
            "num_layers": 4,
            "num_heads": 8,
            "intermediate_size": 1024,
            "max_position_embeddings": 256,
            "dropout": 0.2,
            "position_embedding_type": "rope",
            "activation": "gelu",
            "layer_norm_epsilon": 1e-5,
            "initializer_range": 0.02,
            "tie_word_embeddings": True
        },
        "training": {
            "dataset": dataset,
            "train_samples": train_samples,
            "val_samples": val_samples,
            "batch_size": 64,
            "gradient_accumulation_steps": 4,
            "effective_batch_size": 256,
            "max_steps": 6000,
            "warmup_steps": 600,
            "learning_rate": 5e-5,
            "min_learning_rate": 1e-6,
            "lr_scheduler": "cosine",
            "weight_decay": 0.01,
            "gradient_clip": 0.5,
            "use_bf16": True,
            "optimizer": "adamw",
            "adam_beta1": 0.9,
            "adam_beta2": 0.999,
            "adam_epsilon": 1e-8,
            "max_seq_length": 256
        },
        "data": {
            "tokenizer": "gpt2",
            "dataset_name": dataset_name,
            "dataset_config": "default",
            "text_column": text_column,
            "val_split": val_split,
            "streaming": False,
            "num_workers": 4,
            "prefetch_factor": 2,
            "pin_memory": True
        },
        "logging": {
            "log_dir": log_dir,
            "checkpoint_dir": checkpoint_dir,
            "save_every_steps": 2000,
            "eval_every_steps": 1000,
            "log_every_steps": 50,
            "use_tensorboard": True,
            "use_wandb": False,
            "project_name": f"gptneo-{architecture}-{dataset}",
            "run_name": f"{model_name}_256_4l_{ds_short}"
        },
        "evaluation": {
            "eval_batch_size": 32,
            "eval_max_steps": 100,
            "generation_prompts": [
                "Once upon a time",
                "One day, a little girl",
                "In a big forest",
                "There was a happy dog"
            ],
            "generation_max_length": 100,
            "generation_temperature": 0.8,
            "generation_top_k": 50,
            "generation_top_p": 0.95
        },
        "checkpointing": {
            "save_total_limit": 3,
            "save_best_only": False
        },
        "random_seed": 42
    }

print("Configuration function defined!")

## 3. Training Helper

In [ ]:
import time

training_results = []

def train_model(name, create_fn, config):
    """Train a single model and return the trainer.

    Args:
        name: Display name (e.g. 'MHA-RoPE')
        create_fn: Model creation function (e.g. mha_transformer.create_gptneo_model)
        config: Training configuration dict

    Returns:
        trainer: GPTNeoTrainer instance (for metrics access)
    """
    print("=" * 80)
    print(f"TRAINING: {name}")
    print(f"Dataset: {config['data']['dataset_name']}")
    print("=" * 80)

    # Create model
    model = create_fn(config['model'])
    param_count = model.get_num_params()
    print(f"Parameters: {param_count:,}")

    # Train
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    trainer = GPTNeoTrainer(config, device=device, model=model)

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time

    print(f"\n{name} training complete in {elapsed/60:.1f} minutes")

    # Record results
    training_results.append({
        'name': name,
        'dataset': config['training']['dataset'],
        'params': param_count,
        'time_min': elapsed / 60,
        'checkpoint_dir': config['logging']['checkpoint_dir'],
    })

    # Cleanup
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared")

print("Training helper defined!")

## 4. Train MHA on TinyStories (~35 min)

In [ ]:
mha_ts_config = get_training_config(
    'gptneo_mha_ts', 'gpt_neo_mha', 'tinystories',
    '../logs/gptneo_mha_tinystories', '../checkpoints/gptneo_mha_tinystories')

train_model('MHA-RoPE (TinyStories)', mha_transformer.create_gptneo_model, mha_ts_config)

# Copy best checkpoint
os.makedirs('/content/models/tinystories', exist_ok=True)
src = '/content/checkpoints/gptneo_mha_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mha_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 5. Train MQA on TinyStories (~35 min)

In [ ]:
mqa_ts_config = get_training_config(
    'gptneo_mqa_ts', 'gpt_neo_mqa', 'tinystories',
    '../logs/gptneo_mqa_tinystories', '../checkpoints/gptneo_mqa_tinystories')

train_model('MQA-RoPE (TinyStories)', mqa_transformer.create_gptneo_model, mqa_ts_config)

src = '/content/checkpoints/gptneo_mqa_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mqa_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 6. Train GQA-4 on TinyStories (~35 min)

In [ ]:
gqa_ts_config = get_training_config(
    'gptneo_gqa_ts', 'gpt_neo_gqa', 'tinystories',
    '../logs/gptneo_gqa_tinystories', '../checkpoints/gptneo_gqa_tinystories')
gqa_ts_config['model']['num_kv_heads'] = 4

train_model('GQA-4-RoPE (TinyStories)', gqa_transformer.create_gptneo_model, gqa_ts_config)

src = '/content/checkpoints/gptneo_gqa_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_gqa_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 7. Train MLA on TinyStories (~35 min)

In [ ]:
mla_ts_config = get_training_config(
    'gptneo_mla_ts', 'gpt_neo_mla', 'tinystories',
    '../logs/gptneo_mla_tinystories', '../checkpoints/gptneo_mla_tinystories')
mla_ts_config['model']['d_c'] = 128
mla_ts_config['model']['d_rope'] = 16

train_model('MLA (TinyStories)', mla_transformer.create_gptneo_model, mla_ts_config)

src = '/content/checkpoints/gptneo_mla_tinystories/best_model.pt'
dst = '/content/models/tinystories/best_model_mla_ts_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 8. Save TinyStories Checkpoints to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_ts_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/TinyStories'
os.makedirs(drive_ts_dir, exist_ok=True)

ts_files = [
    'best_model_mha_ts_50pct.pt',
    'best_model_mqa_ts_50pct.pt',
    'best_model_gqa_ts_50pct.pt',
    'best_model_mla_ts_50pct.pt',
]

for f in ts_files:
    src = f'/content/models/tinystories/{f}'
    dst = os.path.join(drive_ts_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  {f} -> Drive ({size_mb:.1f} MB)")
    else:
        print(f"  WARNING: {f} not found")

print(f"\nTinyStories checkpoints saved to: {drive_ts_dir}")

## 9. Train MHA on SimpleStories (~35 min)

In [ ]:
mha_ss_config = get_training_config(
    'gptneo_mha_ss', 'gpt_neo_mha', 'simplestories',
    '../logs/gptneo_mha_simplestories', '../checkpoints/gptneo_mha_simplestories')

train_model('MHA-RoPE (SimpleStories)', mha_transformer.create_gptneo_model, mha_ss_config)

os.makedirs('/content/models/simplestories', exist_ok=True)
src = '/content/checkpoints/gptneo_mha_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mha_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 10. Train MQA on SimpleStories (~35 min)

In [ ]:
mqa_ss_config = get_training_config(
    'gptneo_mqa_ss', 'gpt_neo_mqa', 'simplestories',
    '../logs/gptneo_mqa_simplestories', '../checkpoints/gptneo_mqa_simplestories')

train_model('MQA-RoPE (SimpleStories)', mqa_transformer.create_gptneo_model, mqa_ss_config)

src = '/content/checkpoints/gptneo_mqa_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mqa_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 11. Train GQA-4 on SimpleStories (~35 min)

In [ ]:
gqa_ss_config = get_training_config(
    'gptneo_gqa_ss', 'gpt_neo_gqa', 'simplestories',
    '../logs/gptneo_gqa_simplestories', '../checkpoints/gptneo_gqa_simplestories')
gqa_ss_config['model']['num_kv_heads'] = 4

train_model('GQA-4-RoPE (SimpleStories)', gqa_transformer.create_gptneo_model, gqa_ss_config)

src = '/content/checkpoints/gptneo_gqa_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_gqa_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 12. Train MLA on SimpleStories (~35 min)

In [ ]:
mla_ss_config = get_training_config(
    'gptneo_mla_ss', 'gpt_neo_mla', 'simplestories',
    '../logs/gptneo_mla_simplestories', '../checkpoints/gptneo_mla_simplestories')
mla_ss_config['model']['d_c'] = 128
mla_ss_config['model']['d_rope'] = 16

train_model('MLA (SimpleStories)', mla_transformer.create_gptneo_model, mla_ss_config)

src = '/content/checkpoints/gptneo_mla_simplestories/best_model.pt'
dst = '/content/models/simplestories/best_model_mla_ss_50pct.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Checkpoint saved: {dst}")

## 13. Save SimpleStories Checkpoints to Google Drive

In [ ]:
drive_ss_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/SimpleStories'
os.makedirs(drive_ss_dir, exist_ok=True)

ss_files = [
    'best_model_mha_ss_50pct.pt',
    'best_model_mqa_ss_50pct.pt',
    'best_model_gqa_ss_50pct.pt',
    'best_model_mla_ss_50pct.pt',
]

for f in ss_files:
    src = f'/content/models/simplestories/{f}'
    dst = os.path.join(drive_ss_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  {f} -> Drive ({size_mb:.1f} MB)")
    else:
        print(f"  WARNING: {f} not found")

print(f"\nSimpleStories checkpoints saved to: {drive_ss_dir}")

## 14. Save TensorBoard Logs to Google Drive

In [ ]:
drive_logs_dir = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct/logs'
os.makedirs(drive_logs_dir, exist_ok=True)

log_dirs = [
    'gptneo_mha_tinystories',
    'gptneo_mqa_tinystories',
    'gptneo_gqa_tinystories',
    'gptneo_mla_tinystories',
    'gptneo_mha_simplestories',
    'gptneo_mqa_simplestories',
    'gptneo_gqa_simplestories',
    'gptneo_mla_simplestories',
]

for log_name in log_dirs:
    src = f'/content/logs/{log_name}'
    dst = os.path.join(drive_logs_dir, log_name)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {log_name} -> Drive")
    else:
        print(f"  WARNING: {log_name} not found")

print(f"\nTensorBoard logs saved to: {drive_logs_dir}")

## 15. Final Summary

In [ ]:
total_time = sum(r['time_min'] for r in training_results)

print("TRAINING SUMMARY")
print("=" * 90)
print(f"{'Model':<30} {'Dataset':<15} {'Params':>12} {'Time (min)':>12} {'Checkpoint':>18}")
print("-" * 90)

for r in training_results:
    print(f"{r['name']:<30} {r['dataset']:<15} {r['params']:>12,} {r['time_min']:>12.1f} {'saved':>18}")

print("-" * 90)
print(f"{'TOTAL':>57} {total_time:>12.1f}")
print("=" * 90)

# List all saved files
print("\nSaved Files:")
print("-" * 60)

drive_base = '/content/drive/MyDrive/AttentionHeads_RoPE_50pct'
for root, dirs, files in os.walk(drive_base):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, drive_base)
        print(f"  {rel_path:<50} {size_mb:>8.2f} MB")

print(f"\nAll 8 models trained and saved in {total_time:.1f} minutes ({total_time/60:.1f} hours)")
print(f"Checkpoints: {drive_base}/TinyStories/ and {drive_base}/SimpleStories/")
print(f"Logs: {drive_base}/logs/")